In [4]:
import datetime
import io
import sys
import os
from pathlib import Path
import unicodedata
from datetime import timezone

import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import pybaseball
import requests
import scipy.stats as stats
from bs4 import BeautifulSoup
from pybaseball import statcast
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.calibration import calibration_curve

# Load RosterScraper — Longleaf path
directory = os.path.expanduser('/Users/cbkaplinger/Downloads/MLB-Props/RosterScraper')
sys.path.append(directory)
from RosterScraper import RosterScraper

In [5]:
# StatCast player ID map
url = 'https://docs.google.com/spreadsheets/d/1JgczhD5VDQ1EiXqVG-blttZcVwbZd5_Ne_mefUGwJnk/pub?output=csv'
res = requests.get(url)
ID = pd.read_csv(io.BytesIO(res.content), sep=',')
ID.dropna(subset=['MLBID'], inplace=True)
ID['MLBID'] = ID['MLBID'].astype(int)

# 40-man roster scrape
Rosters = RosterScraper()
BID = Rosters[Rosters['Position'] == 'Batter']
PID = Rosters[Rosters['Position'] == 'Pitcher']

# --- Helper Functions ---
def convert_name(name):
    name_map = {
        'Rockies': 'COL', 'Reds': 'CIN', 'Mariners': 'SEA', 'Nationals': 'WSH',
        'Yankees': 'NYY', 'Astros': 'HOU', 'Red Sox': 'BOS', 'Athletics': 'OAK',
        'Mets': 'NYM', 'Braves': 'ATL', 'Giants': 'SF', 'Brewers': 'MIL',
        'Rays': 'TB', 'Royals': 'KC', 'White Sox': 'CWS', 'Cubs': 'CHC',
        'Angels': 'LAA', 'Tigers': 'DET', 'Diamondbacks': 'ARI', 'Guardians': 'CLE',
        'Orioles': 'BAL', 'Twins': 'MIN', 'Marlins': 'MIA', 'Phillies': 'PHI',
        'Rangers': 'TEX', 'Dodgers': 'LAD', 'Padres': 'SD', 'Pirates': 'PIT',
        'Blue Jays': 'TOR', 'Cardinals': 'STL'
    }
    return name_map.get(name, np.nan)

def flip_names(name):
    if ',' in name:
        last, first = name.split(', ', 1)
        return f"{first} {last}"
    return name

def replace_special_chars(text):
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')

def append_suffix_to_duplicates(df, column):
    seen = {}
    for idx, value in enumerate(df[column]):
        if value in seen:
            seen[value] += 1
            df.at[idx, column] = f"{value}2"
        else:
            seen[value] = 1

In [ ]:
def getDKData2024():
    eastern_time = datetime.datetime.now(timezone.utc).astimezone(timezone(datetime.timedelta(hours=-5)))
    todaysdate = eastern_time.strftime("%m-%d-%Y")
    url = 'https://rotogrinders.com/lineups/mlb?site=draftkings'
    r = requests.get(url)
    soup = BeautifulSoup(r.text, 'lxml')

    gamelist = []
    gamecards = soup.findAll("div", {"class": "game-card-teams"})
    for x in gamecards:
        twoteams = x.findAll("span", {"class": "team-nameplate-mascot"})
        roadteam = convert_name(twoteams[0].text)
        hometeam = convert_name(twoteams[1].text)
        gamekey = "{}@{}".format(roadteam,hometeam)
        gamelist.append(gamekey)

    matchupsdf = pd.DataFrame()
    for game in gamelist:
        roadteam = game.split("@")[0]
        hometeam = game.split("@")[1]
        thisdf1 = pd.DataFrame({"Team": roadteam, "Opp": hometeam, "RoadTeam": roadteam, "HomeTeam": hometeam},index=[0])
        thisdf2 = pd.DataFrame({"Team": hometeam, "Opp": roadteam, "RoadTeam": roadteam, "HomeTeam": hometeam},index=[0])
        matchupsdf = pd.concat([matchupsdf,thisdf1,thisdf2])
        
    oppdict = dict(zip(matchupsdf.Team,matchupsdf.Opp))
    hometeamdict = dict(zip(matchupsdf.Team,matchupsdf.HomeTeam))
    roadteamdict = dict(zip(matchupsdf.Team,matchupsdf.RoadTeam))

    disabled_span_list = []
    for span in soup.findAll("span", {"class": "player-nameplate disabled"}):
        for a in span.findAll("a"):
            disabled_span_list.append(a.text)

    spdata = pd.DataFrame()
    for div in soup.findAll("span", {"class": "player-nameplate", "data-position": "SP"}):
        if "TBD" in str(div):
            playername = "TBD"
            pos = "SP"
            sal = 0
        else:
            for a in div.findAll('a', {'class': 'player-nameplate-name'}):
                playername = a.text.strip()

            strdiv = str(div)
            pos = strdiv[strdiv.find("data-position")+15:strdiv.find("data-salary")-2]
            sal = strdiv[strdiv.find("data-salary")+13:strdiv.find("<div class = 'player-nameplate-info'>")-3]
        try:
            ownership = strdiv[strdiv.find('<span class="small muted" data-auth="502">') + 42:strdiv.find('%')]
            ownership = ownership.replace("</span>", "")
            ownership = ownership.replace("</span", "")
            ownership = ownership.replace("</div>", "")
            ownership = ownership.replace(" ", "")
        except:
            ownership = np.nan

        thisspdata = pd.DataFrame([[playername, sal, ownership]], columns = ["Player", "Salary", "Ownership"])
        spdata = pd.concat([spdata, thisspdata])

    spdata['Player'] = spdata['Player'].replace('Luis Ortiz', 'Luis L. Ortiz')
    spdata['Player'] = spdata['Player'].replace('Mike King', 'Michael King')
    spdata['Player'] = spdata['Player'].replace('Robert Zastryzny', 'Rob Zastryzny')

    spdata2 = pd.merge(spdata, PID[["Name", "Team"]], left_on = ["Player"], right_on = ["Name"], how = "left").rename(columns = {"Team": "PitcherTeam"})
    spdata3 = pd.merge(spdata2, matchupsdf[["Team", "Opp"]], left_on = ["PitcherTeam"], right_on = ["Team"], how = "left").drop(columns = ["Team"])

    append_suffix_to_duplicates(spdata3, 'PitcherTeam')
    append_suffix_to_duplicates(spdata3, 'Opp')

    opp_spname_dict = dict(zip(spdata3.Opp, spdata3.Player))
    opp_spsal_dict = dict(zip(spdata3.Opp, spdata.Salary))
    opp_spown_dict = dict(zip(spdata3.Opp, spdata3.Ownership))

    ludf = pd.DataFrame()
    
    for li in soup.findAll("li", {"class": "lineup-card-player"}):
        for a in li.findAll("a", {"class": ["player-nameplate-name", "player-nameplate disabled"]}):
            playername = a.text

        listring = str(li)
        for span in li.find("span", {"class": "small"}):
            luspot = span.text
            luspot = luspot.replace("\n", "")
            luspot = luspot.strip()
            luspot = int(luspot)
        pos = listring[listring.find("data-position")+15:listring.find("data-salary")-2]
        sal = listring[listring.find("data-salary")+13:listring.find("<span class='small'>")-3]
        ownership = ownership.replace("</span>", "")
        ownership = ownership.replace("</span", "")
        ownership = ownership.replace("</li", "")
        ownership = ownership.replace("</div>", "")
        ownership = ownership.replace(" ", "")

        try:
            sal = int(sal)
        except:
            sal = 0
        thisludf = pd.DataFrame([[playername, luspot, sal, ownership]], columns = ["Player", "Spot", "Sal", "Ownership"])
        ludf = pd.concat([ludf, thisludf])

    ludf2 = pd.merge(ludf, BID[["Name", "Team"]], left_on = ["Player"], right_on = ["Name"], how = "left").rename(columns = {"Team": "BatterTeam"})
    ludf2['BatterTeam'] = ludf2['BatterTeam'].fillna(method='ffill')
    ludf2['HomeTeam'] = ludf2['BatterTeam'].map(hometeamdict)
    ludf2['RoadTeam'] = ludf2['BatterTeam'].map(roadteamdict)

    ludf2_teamlist = list(ludf2["BatterTeam"])

    dhteams = []
    for x in ludf2_teamlist:
        if ludf2_teamlist.count(x) > 11:
            if x in dhteams:
                pass
            else:
                dhteams.append(x)

    extract_dh = ludf2[ludf2["BatterTeam"].isin(dhteams)]
    new_ludf2 = ludf2[~ludf2["BatterTeam"].isin(dhteams)]

    new_team_list = []
    new_home_list = []
    new_road_list = []
    runcounter = 0

    for x, home, road in zip(extract_dh["BatterTeam"].astype(str), 
                         extract_dh["HomeTeam"].astype(str), 
                         extract_dh["RoadTeam"].astype(str)):
        if runcounter < 18:
            new_team_list.append(x)
            new_home_list.append(home)
            new_road_list.append(road)
            runcounter += 1
        else:
            new_team_list.append(x + "2")
            new_home_list.append(home + "2")
            new_road_list.append(road + "2")
            runcounter += 1

    extract_dh["BatterTeam"] = new_team_list
    extract_dh["HomeTeam"] = new_home_list
    extract_dh["RoadTeam"] = new_road_list

    ludf2 = pd.concat([extract_dh, new_ludf2])
    ludf2["Opp"] = ludf2["BatterTeam"].map(oppdict)
    ludf2['SP'] = ludf2['BatterTeam'].map(opp_spname_dict)
    ludf2['SPSal'] = ludf2['BatterTeam'].map(opp_spsal_dict)
    ludf2['SPOwnership'] = ludf2['BatterTeam'].map(opp_spown_dict)
    ludf2['Date'] = todaysdate
    ludf2['Time'] = np.nan

    ludf3 = ludf2[['BatterTeam','RoadTeam','HomeTeam','Time','Spot','Player','Sal','Ownership','Date', "SP"]]

    dkdata = ludf3.copy()

    try:
        checknan = dkdata[["BatterTeam", "SP"]]
        getnans = checknan[["SP"].isna()]
        if len(getnans) == 0:
            nonans = 1
            nanmapdict = {}
        else:
            nonans = 0
            getnans["SP"] = disabled_span_list
            nanmapdict = dict(zip(getnans.Team, getnans.SP))
    except:
        pass

    try:
        dkdata["SP"] = np.where(dkdata["SP"].isna(), dkdata["BatterTeam"].map(nanmapdict), dkdata["SP"])
    except:
        pass
    
    for i in range(1, len(dkdata) - 1):
        if dkdata.loc[i, 'BatterTeam'] != dkdata.loc[i-1, 'BatterTeam']:
            if dkdata.loc[i, 'BatterTeam'] != dkdata.loc[i+1, 'BatterTeam']:
                dkdata.loc[i, 'BatterTeam'] = np.nan
                dkdata.loc[i, 'HomeTeam'] = np.nan
                dkdata.loc[i, 'RoadTeam'] = np.nan
                dkdata.loc[i, 'SP'] = np.nan

    
    dkdata[["BatterTeam", "RoadTeam", "HomeTeam"]] = dkdata[["BatterTeam", "RoadTeam", "HomeTeam"]].fillna(method='ffill')
    dkdata = dkdata.drop_duplicates(subset = ["BatterTeam", "SP"], keep = "first")
    dkdata = dkdata.drop(columns = ["Time", "Sal", "Ownership"])

    dkdata['BatterTeam'] = dkdata['BatterTeam'].replace('ARI', 'AZ')
    dkdata['RoadTeam'] = dkdata['RoadTeam'].replace('ARI', 'AZ')
    dkdata['HomeTeam'] = dkdata['HomeTeam'].replace('ARI', 'AZ')

    dkdata['Date'] = pd.to_datetime(dkdata['Date'])
    dkdata['Date'] = dkdata['Date'].dt.strftime('%Y-%m-%d')
    dkdata = dkdata.set_index("Date")
    dkdata = dkdata[["BatterTeam", "RoadTeam", "HomeTeam", "SP"]]
    dkdata = pl.from_pandas(dkdata)

    return(dkdata)

In [16]:
base_path = Path("/Users/cbkaplinger/Downloads/MLB-Props/Data/Savant-Data/regular").expanduser()
years = [2022, 2023, 2024, 2025]

BAT_TRACKING_COLS = [
    "attack_angle", "attack_direction", "swing_path_tilt",
    "intercept_ball_minus_batter_pos_x_inches",
    "intercept_ball_minus_batter_pos_y_inches",
    "bat_speed", "swing_length",
]

dfs = {}
for year in years:
    year_path = base_path / str(year)
    parquet_files = list(year_path.glob("*.parquet"))
    if parquet_files:
        df = pl.read_parquet(parquet_files[0])
        df = df.drop([c for c in BAT_TRACKING_COLS if c in df.columns])
        # Normalize game_date to a single time unit across all years
        if "game_date" in df.columns:
            df = df.with_columns(pl.col("game_date").cast(pl.Datetime("us")))
        dfs[year] = df

In [17]:
combined = pl.concat(list(dfs.values()), how="diagonal")

# ── Step 2: Derive BatterTeam ──────────────────────────────────────────────
combined = combined.with_columns([
    pl.when(pl.col("inning_topbot") == "Top")
      .then(pl.col("away_team"))
      .otherwise(pl.col("home_team"))
      .alias("BatterTeam"),
    pl.when(pl.col("inning_topbot") == "Top")
      .then(pl.col("home_team"))
      .otherwise(pl.col("away_team"))
      .alias("PitcherTeam"),
])

# ── Step 3: Filter to PA-ending events only ────────────────────────────────
PA_EVENTS = [
    'single','double','triple','home_run','field_out','strikeout','walk',
    'hit_by_pitch','grounded_into_double_play','sac_fly','force_out',
    'fielders_choice','strikeout_double_play','fielders_choice_out',
    'double_play','sac_bunt','other_out','catcher_interf',
]
sc_pa = combined.filter(pl.col("events").is_in(PA_EVENTS))

# ── Step 4: wOBA numerator weights (2023 standard) ────────────────────────
WOBA_WEIGHTS = {
    'walk': 0.690, 'hit_by_pitch': 0.722, 'single': 0.888,
    'double': 1.271, 'triple': 1.616, 'home_run': 2.101,
}
NON_WOBA_DENOM = ['walk', 'hit_by_pitch', 'sac_bunt', 'sac_fly', 'catcher_interf']

sc_pa = sc_pa.with_columns([
    pl.col("events").replace(WOBA_WEIGHTS, default=0.0).cast(pl.Float64).alias("woba_value"),
    pl.col("events").is_in(NON_WOBA_DENOM).cast(pl.Int8).map_elements(
        lambda x: 0 if x else 1, return_dtype=pl.Int8
    ).alias("woba_denom"),
    pl.col("events").is_in(['strikeout','strikeout_double_play']).cast(pl.Int8).alias("is_k"),
    pl.col("events").is_in(['walk']).cast(pl.Int8).alias("is_bb"),
    pl.col("description").is_in(['swinging_strike','swinging_strike_blocked']).cast(pl.Int8).alias("is_swstr"),
    (pl.col("launch_speed") >= 95.0).cast(pl.Int8).alias("is_hard_hit"),
    (pl.col("bb_type") == "ground_ball").cast(pl.Int8).alias("is_gb"),
    (pl.col("bb_type") == "fly_ball").cast(pl.Int8).alias("is_fb"),
    (pl.col("bb_type") == "line_drive").cast(pl.Int8).alias("is_ld"),
    pl.col("description").is_in(['ball','blocked_ball','pitchout']).cast(pl.Int8).alias("is_ball_desc"),
    pl.col("description").is_in(['ball','blocked_ball','pitchout',
                                  'called_strike','swinging_strike',
                                  'swinging_strike_blocked','foul',
                                  'foul_tip']).cast(pl.Int8).alias("is_pitch"),
    pl.lit(1).cast(pl.Int8).alias("pitch_count"),
])

# ── Step 5: Full-game team batting aggregation ─────────────────────────────
game_bat = (
    sc_pa
    .sort(["game_date", "BatterTeam", "at_bat_number"])
    .group_by(["game_date", "BatterTeam", "away_team", "home_team"], maintain_order=True)
    .agg([
        pl.col("at_bat_number").n_unique().alias("PA"),
        pl.col("woba_value").sum().alias("woba_num"),
        # woba_denom: exclude non-AB PA
        pl.col("woba_denom").sum().alias("woba_den"),
        pl.col("is_k").sum().alias("k_count"),
        pl.col("is_bb").sum().alias("bb_count"),
        pl.col("is_swstr").sum().alias("swstr_count"),
        pl.col("pitch_count").sum().alias("total_pitches"),
        pl.col("is_hard_hit").sum().alias("hard_hit_count"),
        pl.col("is_gb").sum().alias("gb_count"),
        pl.col("is_fb").sum().alias("fb_count"),
        pl.col("is_ld").sum().alias("ld_count"),
        # xwOBA/xBA numerators — for PA-weighted rolling
        pl.col("estimated_woba_using_speedangle").mean().alias("xwOBA_raw"),
        pl.col("estimated_ba_using_speedangle").mean().alias("xBA_raw"),
        # innings faced by the starter (proxy: unique innings where starter pitched)
        pl.col("inning").n_unique().alias("innings_total"),
    ])
    .sort(["BatterTeam", "game_date"])
)

print(f"Game-level batting shape: {game_bat.shape}")

/var/folders/bs/6lp27nzn3wvg1ygrkwrcl96w0000gn/T/ipykernel_82258/2981902667.py:32: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("events").replace(WOBA_WEIGHTS, default=0.0).cast(pl.Float64).alias("woba_value"),


Game-level batting shape: (19076, 18)


In [ ]:
# ── Handedness splits (P20 only, smaller samples) ─────────────────────────
# Re-aggregate from pitch level filtered by p_throws
for hand, suffix in [('L', 'vsL'), ('R', 'vsR')]:
    hand_pa = sc_pa.filter(pl.col("p_throws") == hand)
    hand_game = (
        hand_pa
        .group_by(["game_date", "BatterTeam"])
        .agg([
            pl.col("woba_value").sum().alias("woba_num_h"),
            pl.col("woba_denom").sum().alias("woba_den_h"),
            pl.col("at_bat_number").n_unique().alias("pa_h"),
        ])
        .to_pandas()
    )
    hand_game['game_date'] = pd.to_datetime(hand_game['game_date'])

    hand_rolls = []
    for team, grp in hand_game.groupby('BatterTeam'):
        grp = grp.sort_values('game_date').copy()
        grp[f'opp_woba_{suffix}_P20'] = pa_roll_shrunk(
            grp['woba_num_h'], grp['woba_den_h'], 720, LEAGUE['woba'], SHRINK_PA
        )
        hand_rolls.append(grp[['game_date', 'BatterTeam', f'opp_woba_{suffix}_P20']])

    hand_df = pd.concat(hand_rolls)
    opp_df = opp_df.merge(hand_df, on=['game_date', 'BatterTeam'], how='left')
    opp_df[f'opp_woba_{suffix}_P20'] = opp_df[f'opp_woba_{suffix}_P20'].fillna(LEAGUE['woba'])

# ── Filter to 2023+ (2022 was warm-up only) ───────────────────────────────
opp_df = opp_df[opp_df['game_date'] >= '2023-01-01'].copy()
opp_df = opp_df.rename(columns={'BatterTeam': 'opponent_team'})

# ── Output ─────────────────────────────────────────────────────────────────
import platform
if 'longleaf' in platform.node() or 'longleaf' in os.getcwd():
    out_path = Path("/nas/longleaf/home/kapcam/MLB/Data/Batting-Features")
else:
    out_path = Path("~/MLB/Data/Batting-Features").expanduser()

out_path.mkdir(parents=True, exist_ok=True)
opp_df.to_parquet(out_path / "OppBatting2023-2025.parquet", index=False)
opp_df.to_csv(out_path / "OppBatting2023-2025.csv", index=False)
print(f"Saved opponent batting features: {opp_df.shape}")
print(f"Features: {[c for c in opp_df.columns if c.startswith('opp_')]}")